# Analysis of wavefront sensor data from ALS 5.3.1

Date: Created May 2026 

Author: Wei "Francis" He (francisho@lbl.gov)

Based on earlier versions by Dr. Antoine Islegen-Wojdyla and myself. 

___

## Overview

We (Antoine, Henry, and Francis) collected a new set of data (this time with better metadata, and no binning) with various bender settings and photon energy settings. We want to analyze that data.


# SETUP 

## Import library 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from tiled.client import from_uri
import os


In [ ]:
import wfs_utils_2 as wfu

## Access Data from the Server

__Note:__ We tried read data from db with `tiled.__version__='0.2.9'`, but it failed; had to reinstall a 0.2.1 version to make it work. 

In [ ]:
# --- Server connection ---
api_key_path = "/Users/francisho/Documents/TILED_SINGLE_USER_API_KEY.txt"

with open(api_key_path, "r") as f:
    api_key = f.read().strip()

if not api_key:
    raise ValueError("API key file is empty or not found.")

tiled_client = from_uri("http://131.243.80.226:8000", api_key=api_key)

__Important scans files:__

`Uid = '88dd6a7f-0338-446f-8185-fa9e004add04'`: A grid scan for `RE(grid_scan([basler_5472], mono_energy, 7000, 10000, 7, Obender, -2000, -0000, 11, snake_axes=False, md=md))`


In [ ]:
# --- Dataset ---
DATASET_ID = '88dd6a7f-0338-446f-8185-fa9e004add04'
db = tiled_client[DATASET_ID]

# Metadata

wfs_chip_pos_mm = db['baseline']['bcs_linear_motor'].read()
exp_time_s = db.metadata['start']['exposure time [s]']
pixel_size_um = db.metadata['start']['pixel size [um]']
binning = db.metadata['start']['binning']
effective_pixel_size_um = db.metadata['start']['effective pixel size [um]']
camera_model = db.metadata['start']['camera model']
objective_model = db.metadata['start']['objective model']

print(f"Camera:          {camera_model}")
print(f"Objective:       {objective_model}")
print(f"Pixel size:      {pixel_size_um} µm  (binning: {binning}×)")
print(f"Effective pixel: {effective_pixel_size_um} µm")
print(f"Exposure time:   {exp_time_s} s")
print(f"WFS chip position (mm): {wfs_chip_pos_mm}")


In [ ]:
# Arrays (slow) 
images    = db['primary']['basler_5472_image'].read()
energy_eV = db['primary']['mono_energy_energy_eV'].read()
bender_um = db['primary']['bender_um'].read()

print(f"Images shape: {images.shape}  (n_frames × height × width)")
print(f"Energy range: {energy_eV.min():.1f} – {energy_eV.max():.1f} eV")
print(f"Bender range: {bender_um.min():.1f} – {bender_um.max():.1f} µm")

In [ ]:
# Quick visualization of one image
idx = 24

plt.figure(figsize=(20, 8))
plt.imshow(images[idx].T, cmap='hot')
plt.title(f'WFS Image: Frame {idx} | Energy = {energy_eV[idx]:.1f} eV | Bender = {bender_um[idx]:.1f} µm')
plt.colorbar(label='Counts', shrink=0.5)
plt.tight_layout()
plt.show()

# ANALYSIS

## Single File 

Phase recon for image `idx=24`. 


In [ ]:
# --- Physical parameters ---
z_T             = 0.3                              # Grating-to-detector distance [m]
grating_pitch_m = 7e-6                             # Grating pitch from fabrication [m]
dx_m            = effective_pixel_size_um * 1e-6   # Effective pixel size [m]
wavelength_m    = 1.24e-6 / energy_eV[idx]         # X-ray wavelength [m]

print(f"z_T:             {z_T*1e3:.0f} mm")
print(f"Grating pitch:   {grating_pitch_m*1e6:.1f} µm")
print(f"Pixel size:      {dx_m*1e6:.3f} µm")
print(f"Wavelength:      {wavelength_m*1e9:.4f} nm  ({energy_eV[idx]:.1f} eV)")

### Data extraction


In [ ]:
idx = 24
y_sub = 300   # pixels cropped from top for display

fig, ax = plt.subplots(figsize=(20, 5))
ax.imshow(images[idx].T[y_sub:600, 900:2200], cmap='hot')

for y, color in zip([460, 470, 490], ['cyan', 'lime', 'magenta']):
    ax.axhline(y - y_sub, color=color, linewidth=1.5, linestyle='--', label=f'y={y}')

ax.set_title(f'Frame {idx} | Horizontal guide lines')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
y_cuts = [470, 475, 480, 485, 490]#, 465, 495]

fig, ax = plt.subplots(figsize=(10, 8))
for y in y_cuts:
    profile = images[idx][:, y]
    x_m = np.arange(len(profile)) * dx_m
    if y in [465, 495]:
        ax.plot(x_m * 1e6, profile, linewidth=1, alpha = 0.5, linestyle='--', label=f'y={y}')
    else:
        ax.plot(x_m * 1e6, profile, linewidth=1, label=f'y={y}')

ax.set_xlabel('Position [µm]')
ax.set_ylabel('Intensity [counts]')
ax.set_xlim(400, 1100)
ax.legend(fontsize=10)
ax.set_title(f'Frame {idx} | Single-column line cuts')
plt.tight_layout()
plt.show()

In [ ]:
# Plot the 1D profile averaged over the fringe band (y=470–490)

# ROI definition 
x_roi = slice(None)          # full horizontal extent for now
y_roi = slice(470, 490)      # fringe band, confirmed from line cuts above
# y_roi = slice(475, 485)      # fringe band, confirmed from line cuts above --- IGNORE ---

roi = images[idx][:, y_roi]                  # shape: (n_x, 20)
line_profile = np.mean(roi, axis=1)          # average over y → 1D
x_m = np.arange(len(line_profile)) * dx_m

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(x_m * 1e6, line_profile, linewidth=1)
ax.set_xlabel('Position [µm]')
ax.set_ylabel('Intensity [counts]')
# ax.set_xlim(400, 1100)
ax.set_xlim(803, 909)
ax.set_ylim(150, 250)
ax.set_title(f'Frame {idx} | ROI-averaged profile (y=470–490)')
plt.tight_layout()
plt.show()

### Phase reconstruction


In [ ]:
# Trim to beam region to reduce edge noise in FFT
# x_beam = slice(900, 2200)   # pixel range covering ~450–1100 µm
x_beam = slice(0, len(line_profile))   # full range 

# For individual line cuts (same x range)
y_cuts = [470, 475, 480, 485, 490]
profiles = {y: images[idx][x_beam, y] for y in y_cuts}
line_beam = line_profile[x_beam]

# FFT
fft_avg = np.fft.fftshift(np.fft.fft(line_beam))
freq    = np.fft.fftshift(np.fft.fftfreq(len(line_beam), dx_m))

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

ax = axes[0]
ax.semilogy(freq * 1e-3, np.abs(fft_avg), linewidth=1)
ax.set_xlabel('Spatial Frequency [cycles/mm]')
ax.set_ylabel('FFT Magnitude (log)')
ax.set_title('FFT — ROI-averaged profile')
ax.set_xlim(-100, 300)
ax.grid(True, alpha=0.3, which='both')

ax = axes[1]
for y, prof in profiles.items():
    fft_y = np.fft.fftshift(np.fft.fft(prof))
    ax.semilogy(freq * 1e-3, np.abs(fft_y), linewidth=1, alpha=0.7, label=f'y={y}')
ax.set_xlabel('Spatial Frequency [cycles/mm]')
ax.set_ylabel('FFT Magnitude (log)')
ax.set_title('FFT — individual line cuts')
ax.set_xlim(-100, 300)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

In [ ]:
# --- Carrier detection ---
search_mask = (freq >= 100e3) & (freq <= 200e3)
carrier_idx  = np.argmax(np.abs(fft_avg) * search_mask)
k_carrier    = freq[carrier_idx]

print(f"Measured carrier: {k_carrier*1e-3:.2f} cycles/mm")
print(f"Ideal carrier:    {1/grating_pitch_m*1e-3:.2f} cycles/mm")
print(f"Ratio:            {k_carrier/(1/grating_pitch_m):.4f}")

# --- Gaussian filter centered on +1 order ---
filter_sigma  = k_carrier / 5.0   # filter width ~ carrier/5
gauss_filter  = np.exp(-0.5 * ((freq - k_carrier) / filter_sigma)**2)
fft_filtered  = fft_avg * gauss_filter

# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(freq * 1e-3, np.abs(fft_avg),      linewidth=1,   alpha=0.6, label='Original FFT')
ax.semilogy(freq * 1e-3, np.abs(fft_filtered),  linewidth=1.5, label='Filtered')
ax.axvline(k_carrier * 1e-3, color='red', linestyle='--', linewidth=1, label=f'Carrier = {k_carrier*1e-3:.1f} cyc/mm')
ax.set_xlabel('Spatial Frequency [cycles/mm]')
ax.set_ylabel('FFT Magnitude (log)')
ax.set_title('Gaussian filter on +1 order')
ax.set_xlim(-100, 300)
ax.set_ylim(1e-2, np.abs(fft_avg).max()*1.2)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

In [ ]:
# IFFT of filtered spectrum → complex field
complex_field   = np.fft.ifft(np.fft.ifftshift(fft_filtered))

# Wrapped phase
phase_wrapped   = np.angle(complex_field)

# Unwrap and center at midpoint
phase_unwrapped = np.unwrap(phase_wrapped)
phase_centered  = phase_unwrapped - phase_unwrapped[len(phase_unwrapped)//2]

# Spatial coordinates
x_m = np.arange(len(line_beam)) * dx_m
x_m = x_m - x_m[len(x_m)//2]   # center at 0

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

ax = axes[0]
ax.plot(x_m * 1e3, phase_wrapped, linewidth=1)
ax.set_xlabel('Position [mm]')
ax.set_ylabel('Phase [rad]')
ax.set_title('Wrapped phase')
ax.set_ylim(-np.pi, np.pi)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(x_m * 1e3, phase_centered, linewidth=1)
ax.set_xlabel('Position [mm]')
ax.set_ylabel('Phase [rad]')
ax.set_title('Unwrapped & centered phase')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Carrier removal (ideal pitch) ---
carrier_ramp    = 2 * np.pi / grating_pitch_m * x_m
carrier_ramp    = carrier_ramp - carrier_ramp[len(carrier_ramp)//2]
delta_phi       = phase_centered - carrier_ramp

# --- Integration ---
W_temp          = -np.cumsum(delta_phi) * dx_m * grating_pitch_m / wavelength_m / z_T
wavefront_rad   = W_temp - W_temp[len(W_temp)//2]
wavefront_nm    = wavefront_rad * wavelength_m / (2 * np.pi) * 1e9

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

ax = axes[0]
ax.plot(x_m * 1e3, delta_phi, linewidth=1)
ax.set_xlabel('Position [mm]')
ax.set_ylabel('Δφ [rad]')
ax.set_title('Differential phase after carrier removal')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(x_m * 1e3, wavefront_nm, linewidth=1)
ax.set_xlabel('Position [mm]')
ax.set_ylabel('Wavefront [nm OPL]')
ax.set_title('Reconstructed wavefront W(x)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Wavefront PV:  {wavefront_nm.max() - wavefront_nm.min():.2f} nm")
print(f"Wavefront RMS: {np.std(wavefront_nm):.2f} nm")

## Different Bendings at 8000 eV


### Data extraction

The selection of ROI may vary depending on the bending we chose. Thus, in the following we first survey how the image look, and how the ROI choice may affect the FFT spectrum. 

In [ ]:
# --- Visual survey: all 11 bender settings at 8000 eV ---
idx_sweep = range(22, 33)   # 11 frames: bender = −2000 → 0 µm

panels = [images[i].T for i in idx_sweep]
stacked = np.vstack(panels)

fig, ax = plt.subplots(figsize=(10, 15))
ax.imshow(stacked, cmap='inferno', aspect='auto')

# Draw horizontal separators between frames
n_y = panels[0].shape[0]
for k in range(1, len(idx_sweep)):
    ax.axhline(k * n_y - 0.5, color='cyan', linewidth=0.8, linestyle='--')

# Label each panel with its bender value
for k, i in enumerate(idx_sweep):
    ax.text(10, k * n_y + n_y * 0.05,
            f'idx={i}  bender={bender_um[i]:.0f} µm',
            color='white', fontsize=8, va='top')

ax.set_title('Bender sweep at 8000 eV — stacked frames (top: −2000 µm, bottom: 0 µm)')
ax.set_xlabel('x [pixels]')
ax.set_ylabel('y [pixels, repeated per frame]')
plt.tight_layout()
plt.show()

In [ ]:
# --- ROI verification: line cuts for all bender settings ---
idx_sweep = range(22, 33)
y_cuts = [470, 475, 480, 485]
colors = ['C0', 'C1', 'C2', 'C3', 'C4']

fig, axes = plt.subplots(11, 1, figsize=(15, 40))
axes = axes.ravel()

for k, idx in enumerate(idx_sweep):
    ax = axes[k]
    for y, col in zip(y_cuts, colors):
        profile = images[idx][:, y]
        x_um = np.arange(len(profile)) * dx_m * 1e6
        # ax.plot(x_um[750:900], profile[750:900], linewidth=1, color=col, label=f'y={y}')
        ax.plot(x_um, profile, linewidth=1, color=col, label=f'y={y}')
    ax.set_xlim(300, 1100)
    ax.set_xlabel('Position [µm]', fontsize=8)
    ax.set_ylabel('Counts', fontsize=8)
    ax.set_title(f'idx={idx}  bender={bender_um[idx]:.0f} µm', fontsize=9)
    ax.legend(fontsize=7, loc='upper right')
    ax.tick_params(labelsize=7)

# Hide the unused 12th subplot
axes[-1].set_visible(False)

fig.suptitle('ROI verification: y-line cuts at 470, 480, 490 — bender sweep (8000 eV)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- FFT verification: carrier check for all bender settings ---
idx_sweep = range(22, 33)
y_cuts    = [470, 475, 480, 485]
y_roi     = slice(470, 486)

for idx in idx_sweep:
    line_avg = np.mean(images[idx][:, y_roi], axis=1)
    freq     = np.fft.fftshift(np.fft.fftfreq(len(line_avg), dx_m))
    fft_avg  = np.fft.fftshift(np.fft.fft(line_avg))

    # Carrier detection within 120–180 cycles/mm
    search_mask  = (freq >= 120e3) & (freq <= 180e3)
    carrier_idx  = np.argmax(np.abs(fft_avg) * search_mask)
    k_carrier    = freq[carrier_idx]

    fig, axes = plt.subplots(2, 1, figsize=(12, 6))

    for ax in axes:
        ax.axvline(1/grating_pitch_m * 1e-3, color='red',   linestyle='--', linewidth=1, label=f'Ideal: {1/grating_pitch_m*1e-3:.1f} cyc/mm')
        ax.axvline(k_carrier * 1e-3,          color='orange', linestyle='--', linewidth=1, label=f'Measured: {k_carrier*1e-3:.1f} cyc/mm')
        ax.set_xlim(0, 300)
        ax.set_xlabel('Spatial frequency [cycles/mm]')
        ax.set_ylabel('FFT magnitude')
        ax.grid(True, alpha=0.3, which='both')

    axes[0].semilogy(freq * 1e-3, np.abs(fft_avg), linewidth=1.5, color='C0', label='ROI avg (470–485)')
    axes[0].set_title(f'FFT — ROI average | idx={idx}  bender={bender_um[idx]:.0f} µm')
    axes[0].legend(fontsize=8)

    for y in y_cuts:
        prof  = images[idx][:, y]
        fft_y = np.fft.fftshift(np.fft.fft(prof))
        axes[1].semilogy(freq * 1e-3, np.abs(fft_y), linewidth=1, alpha=0.7, label=f'y={y}')
    axes[1].set_title(f'FFT — individual cuts | idx={idx}  bender={bender_um[idx]:.0f} µm')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

### Phase reconstruction

In [ ]:
# Carrier detection and Gaussian filter 
idx_sweep = range(22, 33)
y_roi     = slice(470, 485)
rec       = {}

for idx in idx_sweep:
    line_avg = np.mean(images[idx][:, y_roi], axis=1)
    N        = len(line_avg)
    freq     = np.fft.fftshift(np.fft.fftfreq(N, dx_m))
    fft_avg  = np.fft.fftshift(np.fft.fft(line_avg))

    search_mask  = (freq >= 120e3) & (freq <= 180e3)
    carrier_idx  = np.argmax(np.abs(fft_avg) * search_mask)
    k_carrier    = freq[carrier_idx]

    sigma        = k_carrier / 5.0
    gauss_filt   = np.exp(-0.5 * ((freq - k_carrier) / sigma)**2)
    fft_filtered = fft_avg * gauss_filt

    rec[idx] = dict(freq=freq, fft_avg=fft_avg,
                    fft_filtered=fft_filtered, k_carrier=k_carrier)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.semilogy(freq * 1e-3, np.abs(fft_avg),      linewidth=1,   alpha=0.6, label='Original')
    ax.semilogy(freq * 1e-3, np.abs(fft_filtered), linewidth=1.5, label='Filtered')
    ax.axvline(1/grating_pitch_m * 1e-3, color='red',    linestyle='--', linewidth=1, label=f'Ideal: {1/grating_pitch_m*1e-3:.1f} cyc/mm')
    ax.axvline(k_carrier * 1e-3,          color='orange', linestyle='--', linewidth=1, label=f'Measured: {k_carrier*1e-3:.1f} cyc/mm')
    ax.set_xlim(0, 300)
    ax.set_ylim(1e-2, np.abs(fft_avg).max() * 1.2)
    ax.set_xlabel('Spatial frequency [cycles/mm]')
    ax.set_ylabel('FFT magnitude')
    ax.set_title(f'FFT + filter | idx={idx}  bender={bender_um[idx]:.0f} µm')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()

# Carrier frequency summary table
k_ideal = 1 / grating_pitch_m
print(f"{'idx':>5}  {'bender (µm)':>12}  {'k_meas (cyc/mm)':>17}")
print("-" * 40)
for idx in idx_sweep:
    k = rec[idx]['k_carrier']
    print(f"{idx:>5}  {bender_um[idx]:>12.0f}  {k*1e-3:>17.2f}")

In [ ]:
# ifft + unwrapping + carrier removal + integration → wavefront 

for idx in idx_sweep:
    phase_centered = rec[idx]['fft_filtered']
    N              = len(phase_centered)

    complex_field   = np.fft.ifft(np.fft.ifftshift(rec[idx]['fft_filtered']))
    phase_unwrapped = np.unwrap(np.angle(complex_field))
    phase_centered  = phase_unwrapped - phase_unwrapped[N // 2]
    x_m             = (np.arange(N) - N // 2) * dx_m

    carrier_ramp  = 2 * np.pi / grating_pitch_m * x_m
    carrier_ramp  = carrier_ramp - carrier_ramp[N // 2]
    delta_phi     = phase_centered - carrier_ramp

    W_temp        = -np.cumsum(delta_phi) * dx_m * grating_pitch_m / wavelength_m / z_T
    wavefront_rad = W_temp - W_temp[N // 2]
    wavefront_nm  = wavefront_rad * wavelength_m / (2 * np.pi) * 1e9

    rec[idx]['x_m']           = x_m
    rec[idx]['delta_phi']     = delta_phi
    rec[idx]['wavefront_rad'] = wavefront_rad
    rec[idx]['wavefront_nm']  = wavefront_nm

    fig, axes = plt.subplots(2, 1, figsize=(12, 7))

    ax = axes[0]
    ax.plot(x_m * 1e3, delta_phi, linewidth=1)
    ax.set_xlabel('Position [mm]')
    ax.set_ylabel('Δφ [rad]')
    ax.set_title(f'Differential phase | idx={idx}  bender={bender_um[idx]:.0f} µm')
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(x_m * 1e3, wavefront_nm, linewidth=1)
    ax.set_xlabel('Position [mm]')
    ax.set_ylabel('Wavefront [nm OPL]')
    ax.set_title(f'Reconstructed wavefront | idx={idx}  bender={bender_um[idx]:.0f} µm')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"idx={idx}  bender={bender_um[idx]:.0f} µm  |  PV={wavefront_nm.max()-wavefront_nm.min():.2f} nm  RMS={np.std(wavefront_nm):.2f} nm")

In [ ]:
# --- C4: Overlay of reconstructed wavefronts (three groups) ---
groups = [list(idx_sweep)[:4], list(idx_sweep)[4:7], list(idx_sweep)[7:]]

for group in groups:
    fig, ax = plt.subplots(figsize=(12, 5))
    for idx in group:
        x_m          = rec[idx]['x_m']
        wavefront_nm = rec[idx]['wavefront_nm']
        ax.plot(x_m * 1e3, wavefront_nm, linewidth=1,
                label=f'idx={idx}  bender={bender_um[idx]:.0f} µm')
    ax.set_xlim(-0.5, 0.5)
    ax.set_ylim(-85, 55)
    ax.set_xlabel('Position [mm]')
    ax.set_ylabel('Wavefront [nm OPL]')
    ax.set_title('Reconstructed wavefronts — bender sweep at 8000 eV')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### Extra attempts

1. Detrending
2. Effects of filter selection 

#### Detrending

Not really useful in our case here. 

In [ ]:
# Detrended wavefronts 
groups = [list(idx_sweep)[:4], list(idx_sweep)[4:7], list(idx_sweep)[7:]]

for group in groups:
    fig, ax = plt.subplots(figsize=(12, 5))
    for idx in group:
        x_m          = rec[idx]['x_m']
        wavefront_rad = rec[idx]['wavefront_rad']

        _, aberrations_rad, _ = wfu.detrending(wavefront_rad, x_m, fit_order=2)
        aberrations_nm = aberrations_rad * rec[idx].get('wavelength_m', wavelength_m) / (2 * np.pi) * 1e9

        ax.plot(x_m * 1e3, aberrations_nm, linewidth=1,
                label=f'idx={idx}  bender={bender_um[idx]:.0f} µm')

    ax.set_xlim(-0.5, 0.5)
    ax.set_xlabel('Position [mm]')
    ax.set_ylabel('Residual aberration [nm OPL]')
    ax.set_title('Detrended wavefronts (parabola removed) — bender sweep at 8000 eV')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

#### Change the filter `sigma`? 

Currently we are using `sigma = k_carrier/5`; what if `sigma = k_carrier/10`? 


In [ ]:
# --- C_sharp_1: Rerun carrier filter with sharper sigma ---
sigma_ratio_sharp = 10   # compare to default of 5
rec_sharp = {}

for idx in idx_sweep:
    line_avg = np.mean(images[idx][:, y_roi], axis=1)
    N        = len(line_avg)
    freq     = np.fft.fftshift(np.fft.fftfreq(N, dx_m))
    fft_avg  = np.fft.fftshift(np.fft.fft(line_avg))

    search_mask  = (freq >= 120e3) & (freq <= 180e3)
    carrier_idx  = np.argmax(np.abs(fft_avg) * search_mask)
    k_carrier    = freq[carrier_idx]

    sigma        = k_carrier / sigma_ratio_sharp
    gauss_filt   = np.exp(-0.5 * ((freq - k_carrier) / sigma)**2)
    fft_filtered = fft_avg * gauss_filt

    rec_sharp[idx] = dict(freq=freq, fft_avg=fft_avg,
                          fft_filtered=fft_filtered, k_carrier=k_carrier)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.semilogy(freq * 1e-3, np.abs(fft_avg),                        linewidth=1,   alpha=0.5, label='Original')
    ax.semilogy(freq * 1e-3, np.abs(rec[idx]['fft_filtered']),        linewidth=1.5, alpha=0.7, label=f'σ = k/5  (default)')
    ax.semilogy(freq * 1e-3, np.abs(fft_filtered),                    linewidth=1.5, label=f'σ = k/{sigma_ratio_sharp}  (sharp)')
    ax.axvline(k_carrier * 1e-3, color='orange', linestyle='--', linewidth=1, label=f'Carrier: {k_carrier*1e-3:.1f} cyc/mm')
    ax.set_xlim(0, 300)
    ax.set_ylim(1e-2, np.abs(fft_avg).max() * 1.2)
    ax.set_xlabel('Spatial frequency [cycles/mm]')
    ax.set_ylabel('FFT magnitude')
    ax.set_title(f'Filter comparison | idx={idx}  bender={bender_um[idx]:.0f} µm')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()

In [ ]:
# --- C_sharp_2: Reconstruction with sharp filter + overlay ---
groups = [list(idx_sweep)[:4], list(idx_sweep)[4:7], list(idx_sweep)[7:]]

for group in groups:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    for idx in group:
        N   = len(rec_sharp[idx]['fft_filtered'])
        x_m = (np.arange(N) - N // 2) * dx_m

        for r, label, ax in zip([rec[idx], rec_sharp[idx]],
                                 ['σ=k/5', f'σ=k/{sigma_ratio_sharp}'],
                                 axes):
            cf          = np.fft.ifft(np.fft.ifftshift(r['fft_filtered']))
            pu          = np.unwrap(np.angle(cf))
            pc          = pu - pu[N // 2]
            cr          = 2 * np.pi / grating_pitch_m * x_m
            cr          = cr - cr[N // 2]
            dp          = pc - cr
            W           = -np.cumsum(dp) * dx_m * grating_pitch_m / wavelength_m / z_T
            wf_nm       = (W - W[N // 2]) * wavelength_m / (2 * np.pi) * 1e9
            ax.plot(x_m * 1e3, wf_nm, linewidth=1,
                    label=f'idx={idx}  bender={bender_um[idx]:.0f} µm')

    for ax, label in zip(axes, ['σ = k/5  (default)', f'σ = k/{sigma_ratio_sharp}  (sharp)']):
        ax.set_xlim(-0.5, 0.5)
        ax.set_ylim(-90, 30)
        ax.set_ylabel('Wavefront [nm OPL]')
        ax.set_title(f'Bender sweep — {label}')
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Position [mm]')
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Filter check: amplitude envelope vs raw beam profile ---
# Using idx=24 (8000 eV, bender=−1600 µm) as representative frame

idx_check = 24
line_avg  = np.mean(images[idx_check][:, y_roi], axis=1)
N         = len(line_avg)
x_m       = (np.arange(N) - N // 2) * dx_m
freq      = np.fft.fftshift(np.fft.fftfreq(N, dx_m))
fft_avg   = np.fft.fftshift(np.fft.fft(line_avg))

search_mask = (freq >= 120e3) & (freq <= 180e3)
carrier_idx = np.argmax(np.abs(fft_avg) * search_mask)
k_carrier   = freq[carrier_idx]

fig, ax = plt.subplots(figsize=(12, 5))

# Raw beam profile (normalized)
ax.plot(x_m * 1e3, line_avg / line_avg.max(),
        linewidth=1, color='black', alpha=0.5, label='Raw profile (normalized)')

# Amplitude envelope for each sigma
for sigma_ratio, color in [(5, 'C0'), (10, 'C1')]:
    sigma        = k_carrier / sigma_ratio
    gauss_filt   = np.exp(-0.5 * ((freq - k_carrier) / sigma)**2)
    fft_filtered = fft_avg * gauss_filt
    envelope     = np.abs(np.fft.ifft(np.fft.ifftshift(fft_filtered)))
    ax.plot(x_m * 1e3, envelope / envelope.max(),
            linewidth=1.5, color=color, label=f'Envelope  σ = k/{sigma_ratio}')

ax.set_xlim(-1, 1)
ax.set_xlabel('Position [mm]')
ax.set_ylabel('Normalized amplitude')
ax.set_title(f'Amplitude envelope vs raw profile | idx={idx_check}  E={energy_eV[idx_check]:.0f} eV  bender={bender_um[idx_check]:.0f} µm')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Different Energies with Bender` = -1600 um`

### Data extraction


In [ ]:
# --- Multi-energy comparison: fixed bender = −1600 µm (bender_idx=2) ---
idx_sweep    = [2, 13, 24, 35, 46, 57, 68]     # energy_block * 11 + 2
# y_roi        = slice(470, 485)                 # to be re-verified below

# visual survey 
panels  = [images[i].T for i in idx_sweep]
stacked = np.vstack(panels)

fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(stacked, cmap='inferno', aspect='auto')

n_y = panels[0].shape[0]
for k in range(1, len(idx_sweep)):
    ax.axhline(k * n_y - 0.5, color='cyan', linewidth=0.8, linestyle='--')

for k, i in enumerate(idx_sweep):
    ax.text(10, k * n_y + n_y * 0.05,
            f'idx={i}  E={energy_eV[i]:.0f} eV',
            color='white', fontsize=8, va='top')

ax.set_title('Energy sweep at bender = −1600 µm — stacked frames')
ax.set_xlabel('x [pixels]')
ax.set_ylabel('y [pixels, repeated per frame]')
plt.tight_layout()
plt.show()

In [ ]:
# --- E1: ROI line cut verification ---
y_cuts = [470, 475, 480, 485]
colors = ['C0', 'C1', 'C2', 'C3']

fig, axes = plt.subplots(7, 1, figsize=(15, 28))

for k, idx in enumerate(idx_sweep):
    ax = axes[k]
    for y, col in zip(y_cuts, colors):
        profile = images[idx][:, y]
        x_um    = np.arange(len(profile)) * dx_m * 1e6
        ax.plot(x_um, profile, linewidth=1, color=col, label=f'y={y}')
    ax.set_xlim(300+50*k, 1000+50*k)
    ax.set_xlabel('Position [µm]', fontsize=8)
    ax.set_ylabel('Counts', fontsize=8)
    ax.set_title(f'idx={idx}  E={energy_eV[idx]:.0f} eV  bender={bender_um[idx]:.0f} µm', fontsize=9)
    ax.legend(fontsize=7, loc='upper right')
    ax.tick_params(labelsize=7)

fig.suptitle('ROI verification — energy sweep at bender = −1600 µm', fontsize=12, y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# --- E1.5: FFT carrier verification — energy sweep ---
y_cuts = [470, 475, 480, 485]
y_roi  = slice(470, 486)

for idx in idx_sweep:
    line_avg = np.mean(images[idx][:, y_roi], axis=1)
    freq     = np.fft.fftshift(np.fft.fftfreq(len(line_avg), dx_m))
    fft_avg  = np.fft.fftshift(np.fft.fft(line_avg))

    search_mask = (freq >= 120e3) & (freq <= 180e3)
    carrier_idx = np.argmax(np.abs(fft_avg) * search_mask)
    k_carrier   = freq[carrier_idx]

    fig, axes = plt.subplots(2, 1, figsize=(15, 7))

    for ax in axes:
        ax.axvline(1/grating_pitch_m * 1e-3, color='red',    linestyle='--', linewidth=1, label=f'Ideal: {1/grating_pitch_m*1e-3:.1f} cyc/mm')
        ax.axvline(k_carrier * 1e-3,          color='orange', linestyle='--', linewidth=1, label=f'Measured: {k_carrier*1e-3:.1f} cyc/mm')
        ax.set_xlim(0, 300)
        ax.set_ylim(1, np.abs(fft_avg).max() * 1.2)
        ax.set_xlabel('Spatial frequency [cycles/mm]')
        ax.set_ylabel('FFT magnitude')
        ax.grid(True, alpha=0.3, which='both')

    axes[0].semilogy(freq * 1e-3, np.abs(fft_avg), linewidth=1.5, color='C0', label='ROI avg (470–485)')
    axes[0].set_title(f'FFT — ROI average | idx={idx}  E={energy_eV[idx]:.0f} eV')
    axes[0].legend(fontsize=8)

    for y in y_cuts:
        prof  = images[idx][:, y]
        fft_y = np.fft.fftshift(np.fft.fft(prof))
        axes[1].semilogy(freq * 1e-3, np.abs(fft_y), linewidth=1, alpha=0.7, label=f'y={y}')
    axes[1].set_title(f'FFT — individual cuts | idx={idx}  E={energy_eV[idx]:.0f} eV')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

### Phase reconstruction


In [ ]:
# --- E2: Carrier detection and Gaussian filter ---
rec_E = {}

for idx in idx_sweep:
    wavelength_m = 1.24e-6 / energy_eV[idx]   # per-frame wavelength

    line_avg = np.mean(images[idx][:, y_roi], axis=1)
    N        = len(line_avg)
    freq     = np.fft.fftshift(np.fft.fftfreq(N, dx_m))
    fft_avg  = np.fft.fftshift(np.fft.fft(line_avg))

    search_mask  = (freq >= 120e3) & (freq <= 180e3)
    carrier_idx  = np.argmax(np.abs(fft_avg) * search_mask)
    k_carrier    = freq[carrier_idx]

    sigma        = k_carrier / 5.0
    gauss_filt   = np.exp(-0.5 * ((freq - k_carrier) / sigma)**2)
    fft_filtered = fft_avg * gauss_filt

    rec_E[idx] = dict(freq=freq, fft_avg=fft_avg,
                      fft_filtered=fft_filtered, k_carrier=k_carrier,
                      wavelength_m=wavelength_m)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.semilogy(freq * 1e-3, np.abs(fft_avg),      linewidth=1,   alpha=0.6, label='Original')
    ax.semilogy(freq * 1e-3, np.abs(fft_filtered), linewidth=1.5, label='Filtered')
    ax.axvline(1/grating_pitch_m * 1e-3, color='red',    linestyle='--', linewidth=1, label=f'Ideal: {1/grating_pitch_m*1e-3:.1f} cyc/mm')
    ax.axvline(k_carrier * 1e-3,          color='orange', linestyle='--', linewidth=1, label=f'Measured: {k_carrier*1e-3:.1f} cyc/mm')
    ax.set_xlim(0, 300)
    ax.set_ylim(1e-2, np.abs(fft_avg).max() * 1.2)
    ax.set_xlabel('Spatial frequency [cycles/mm]')
    ax.set_ylabel('FFT magnitude')
    ax.set_title(f'FFT + filter | idx={idx}  E={energy_eV[idx]:.0f} eV')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()

k_ideal = 1 / grating_pitch_m
print(f"{'idx':>5}  {'energy (eV)':>12}  {'k_meas (cyc/mm)':>17}  {'ratio':>7}")
print("-" * 48)
for idx in idx_sweep:
    k = rec_E[idx]['k_carrier']
    print(f"{idx:>5}  {energy_eV[idx]:>12.0f}  {k*1e-3:>17.2f}  {k/k_ideal:>7.4f}")

In [ ]:
# --- E3: IFFT + carrier removal + integration ---

for idx in idx_sweep:
    wavelength_m = rec_E[idx]['wavelength_m']   # per-frame
    N            = len(rec_E[idx]['fft_filtered'])

    complex_field   = np.fft.ifft(np.fft.ifftshift(rec_E[idx]['fft_filtered']))
    phase_unwrapped = np.unwrap(np.angle(complex_field))
    phase_centered  = phase_unwrapped - phase_unwrapped[N // 2]
    x_m             = (np.arange(N) - N // 2) * dx_m

    carrier_ramp  = 2 * np.pi / grating_pitch_m * x_m
    carrier_ramp  = carrier_ramp - carrier_ramp[N // 2]
    delta_phi     = phase_centered - carrier_ramp

    W_temp        = -np.cumsum(delta_phi) * dx_m * grating_pitch_m / wavelength_m / z_T
    wavefront_rad = W_temp - W_temp[N // 2]
    wavefront_nm  = wavefront_rad * wavelength_m / (2 * np.pi) * 1e9

    rec_E[idx]['x_m']            = x_m
    rec_E[idx]['delta_phi']      = delta_phi
    rec_E[idx]['wavefront_rad']  = wavefront_rad
    rec_E[idx]['wavefront_nm']   = wavefront_nm

    fig, axes = plt.subplots(2, 1, figsize=(12, 7))

    axes[0].plot(x_m * 1e3, delta_phi, linewidth=1)
    axes[0].set_xlabel('Position [mm]')
    axes[0].set_ylabel('Δφ [rad]')
    axes[0].set_title(f'Differential phase | idx={idx}  E={energy_eV[idx]:.0f} eV')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(x_m * 1e3, wavefront_nm, linewidth=1)
    axes[1].set_xlabel('Position [mm]')
    axes[1].set_ylabel('Wavefront [nm OPL]')
    axes[1].set_title(f'Reconstructed wavefront | idx={idx}  E={energy_eV[idx]:.0f} eV')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"idx={idx}  E={energy_eV[idx]:.0f} eV  |  PV={wavefront_nm.max()-wavefront_nm.min():.2f} nm  RMS={np.std(wavefront_nm):.2f} nm")

In [ ]:
# --- E4: Overlay ---

fig, ax = plt.subplots(figsize=(12, 5))
for idx in idx_sweep:
    x_m          = rec_E[idx]['x_m']
    wavefront_nm = rec_E[idx]['wavefront_nm']
    ax.plot(x_m * 1e3, wavefront_nm, linewidth=1,
            label=f'idx={idx}  E={energy_eV[idx]:.0f} eV')

ax.set_xlim(-0.5, 0.5)
ax.set_ylim(-20, 20)
ax.set_xlabel('Position [mm]')
ax.set_ylabel('Wavefront [nm OPL]')
ax.set_title('Reconstructed wavefronts overlayed (-0.5—0.5 mm) — energy sweep at bender = −1600 µm')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
for idx in idx_sweep:
    x_m          = rec_E[idx]['x_m']
    wavefront_nm = rec_E[idx]['wavefront_nm']
    ax.plot(x_m * 1e3, wavefront_nm, linewidth=1,
            label=f'idx={idx}  E={energy_eV[idx]:.0f} eV')

ax.set_xlim(-0.25, 0.25)
ax.set_ylim(-3, 6)
ax.set_xlabel('Position [mm]')
ax.set_ylabel('Wavefront [nm OPL]')
ax.set_title('Reconstructed wavefronts overlayed (-0.25—0.25 mm) — energy sweep at bender = −1600 µm')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Extra attempts

1. Detrending
2. Effects of filter selection 

#### Detrending

Not really useful in our case here. 

In [ ]:
# Detrended wavefronts — energy sweep 
fig, ax = plt.subplots(figsize=(12, 5))

for idx in idx_sweep:
    x_m           = rec_E[idx]['x_m']
    wavefront_rad = rec_E[idx]['wavefront_rad']
    wavelength_m  = rec_E[idx]['wavelength_m']

    _, aberrations_rad, _ = wfu.detrending(wavefront_rad, x_m, fit_order=2)
    aberrations_nm = aberrations_rad * wavelength_m / (2 * np.pi) * 1e9

    ax.plot(x_m * 1e3, aberrations_nm, linewidth=1,
            label=f'idx={idx}  E={energy_eV[idx]:.0f} eV')

ax.set_xlim(-0.5, 0.5)
ax.set_xlabel('Position [mm]')
ax.set_ylabel('Residual aberration [nm OPL]')
ax.set_title('Detrended wavefronts (parabola removed) — energy sweep at bender = −1600 µm')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Change the filter `sigma`? 

Currently we are using `sigma = k_carrier/5`; what if `sigma = k_carrier/10`? 


In [ ]:
# --- E_sharp_1: Rerun carrier filter with sharper sigma — energy sweep ---
rec_E_sharp = {}

for idx in [2, 13, 24, 35, 46, 57, 68]:
    line_avg = np.mean(images[idx][:, y_roi], axis=1)
    N        = len(line_avg)
    freq     = np.fft.fftshift(np.fft.fftfreq(N, dx_m))
    fft_avg  = np.fft.fftshift(np.fft.fft(line_avg))

    search_mask  = (freq >= 120e3) & (freq <= 180e3)
    carrier_idx  = np.argmax(np.abs(fft_avg) * search_mask)
    k_carrier    = freq[carrier_idx]

    sigma        = k_carrier / sigma_ratio_sharp
    gauss_filt   = np.exp(-0.5 * ((freq - k_carrier) / sigma)**2)
    fft_filtered = fft_avg * gauss_filt

    rec_E_sharp[idx] = dict(freq=freq, fft_avg=fft_avg,
                             fft_filtered=fft_filtered, k_carrier=k_carrier,
                             wavelength_m=1.24e-6 / energy_eV[idx])

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.semilogy(freq * 1e-3, np.abs(fft_avg),                          linewidth=1,   alpha=0.5, label='Original')
    ax.semilogy(freq * 1e-3, np.abs(rec_E[idx]['fft_filtered']),        linewidth=1.5, alpha=0.7, label='σ = k/5  (default)')
    ax.semilogy(freq * 1e-3, np.abs(fft_filtered),                      linewidth=1.5, label=f'σ = k/{sigma_ratio_sharp}  (sharp)')
    ax.axvline(k_carrier * 1e-3, color='orange', linestyle='--', linewidth=1, label=f'Carrier: {k_carrier*1e-3:.1f} cyc/mm')
    ax.set_xlim(0, 300)
    ax.set_ylim(1e-2, np.abs(fft_avg).max() * 1.2)
    ax.set_xlabel('Spatial frequency [cycles/mm]')
    ax.set_ylabel('FFT magnitude')
    ax.set_title(f'Filter comparison | idx={idx}  E={energy_eV[idx]:.0f} eV')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()

In [ ]:
# --- E_sharp_2: Reconstruction with sharp filter + overlay — energy sweep ---
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

for idx in [2, 13, 24, 35, 46, 57, 68]:
    N            = len(rec_E_sharp[idx]['fft_filtered'])
    x_m          = (np.arange(N) - N // 2) * dx_m

    for r, label, ax in zip([rec_E[idx], rec_E_sharp[idx]],
                             ['σ=k/5', f'σ=k/{sigma_ratio_sharp}'],
                             axes):
        wavelength_m = r['wavelength_m']
        cf           = np.fft.ifft(np.fft.ifftshift(r['fft_filtered']))
        pu           = np.unwrap(np.angle(cf))
        pc           = pu - pu[N // 2]
        cr           = 2 * np.pi / grating_pitch_m * x_m
        cr           = cr - cr[N // 2]
        dp           = pc - cr
        W            = -np.cumsum(dp) * dx_m * grating_pitch_m / wavelength_m / z_T
        wf_nm        = (W - W[N // 2]) * wavelength_m / (2 * np.pi) * 1e9
        ax.plot(x_m * 1e3, wf_nm, linewidth=1,
                label=f'idx={idx}  E={energy_eV[idx]:.0f} eV')

for ax, label in zip(axes, ['σ = k/5  (default)', f'σ = k/{sigma_ratio_sharp}  (sharp)']):
    ax.set_xlim(-0.5, 0.5)
    ax.set_ylabel('Wavefront [nm OPL]')
    ax.set_title(f'Energy sweep — {label}')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Position [mm]')
plt.tight_layout()
plt.show()

In [ ]:
# --- Filter check: amplitude envelope vs raw beam profile — energy sweep ---

fig_list = []
for idx in [2, 13, 24, 35, 46, 57, 68]:
    line_avg = np.mean(images[idx][:, y_roi], axis=1)
    N        = len(line_avg)
    x_m      = (np.arange(N) - N // 2) * dx_m
    freq     = np.fft.fftshift(np.fft.fftfreq(N, dx_m))
    fft_avg  = np.fft.fftshift(np.fft.fft(line_avg))

    search_mask = (freq >= 120e3) & (freq <= 180e3)
    carrier_idx = np.argmax(np.abs(fft_avg) * search_mask)
    k_carrier   = freq[carrier_idx]

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(x_m * 1e3, line_avg / line_avg.max(),
            linewidth=1, color='black', alpha=0.5, label='Raw profile (normalized)')

    for sigma_ratio, color in [(5, 'C0'), (10, 'C1')]:
        sigma        = k_carrier / sigma_ratio
        gauss_filt   = np.exp(-0.5 * ((freq - k_carrier) / sigma)**2)
        fft_filtered = fft_avg * gauss_filt
        envelope     = np.abs(np.fft.ifft(np.fft.ifftshift(fft_filtered)))
        ax.plot(x_m * 1e3, envelope / envelope.max(),
                linewidth=1.5, color=color, label=f'Envelope  σ = k/{sigma_ratio}')

    ax.set_xlim(-1, 1)
    ax.set_xlabel('Position [mm]')
    ax.set_ylabel('Normalized amplitude')
    ax.set_title(f'Amplitude envelope vs raw profile | idx={idx}  E={energy_eV[idx]:.0f} eV  bender={bender_um[idx]:.0f} µm')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# GRID ANALYSIS

___

## Comments from Analysis Above

__Fixed parameters__
- `z_T = 0.3` m, `grating_pitch_m = 7e-6` m, `dx_m = effective_pixel_size_um * 1e-6`
- `wavelength_m = 1.24e-6 / energy_eV[idx]` — must be computed **per frame** (varies with energy)

__ROI__
- `y_roi = slice(470, 486)` — validated across all 11 bender settings at 8000 eV and all 7 energies at bender = −1600 µm; y-line cuts overlap cleanly throughout

__Carrier detection__
- Search band: `120–180 cycles/mm`; peak found via `argmax(|FFT| * search_mask)`
- Measured carrier ~144 cyc/mm, consistent across energies (geometry-fixed)
- Use **ideal pitch** `p = 7 µm` for carrier ramp removal; measured `k_carrier` used only for filter centering

__Gaussian filter__
- `sigma_ratio = 10` (i.e. `σ = k_carrier / 10`) — validated by amplitude envelope check
- `sigma_ratio = 5` produces artificial amplitude nulls inside the beam due to insufficient DC/sideband isolation; **do not use**

__Reconstruction formula__
```python
carrier_ramp  = 2 * np.pi / grating_pitch_m * x_m
carrier_ramp  = carrier_ramp - carrier_ramp[N // 2]
delta_phi     = phase_centered - carrier_ramp
W_temp        = -np.cumsum(delta_phi) * dx_m * grating_pitch_m / wavelength_m / z_T
wavefront_rad = W_temp - W_temp[N // 2]
wavefront_nm  = wavefront_rad * wavelength_m / (2 * np.pi) * 1e9
```

__Known problem frames__
- idx = 26, 27, 28 (bender = −1200, −1000, −800 µm at 8000 eV): near-focus, no carrier peak, reconstruction meaningless — flag but do not exclude from loop

__Skipped for now__
- Detrending (parabola removal): implemented in `wfu.detrending()` but not appears to be useful
- Beam masking / beam-region trimming: edges are noisy but not masked

In [ ]:
images.shape

# Old cells (archived)

In [ ]:
x_m = (np.arange(images[idx].shape[0])) * effective_pixel_size_um *1e-6
line_profile = images[idx][:,495]
plt.plot(x_m*1e6, line_profile)
plt.xlabel('vertical position (um)')
plt.ylabel('Intensity')
plt.title('Line profile of WFS image at energy = {:1.1f} eV, bender = {:1.1f} um'.format(energy_eV[idx], bender_um[idx]))
plt.xlim(0, x_m[-1]*1e6)
plt.ylim(0, line_profile.max()*1.1)
plt.show()


In [ ]:
#other useful metadata:
grating_pitch_m = 7e-6  # 
chip_ccd_distance = 300e-3


grating_freq_cpm = 1/grating_pitch_m  # cycles per micron for 7 um pitch
wavelength_m = 1.24e-6 / (energy_eV[idx])  # Convert eV to wavelength in meters
px_m = effective_pixel_size_um * 1e-6  # Convert effective pixel size to meters

line = line_profile
line_FFT = np.fft.fftshift(np.fft.fft(line, axis=0), axes=0)
freq_y_cpm = np.fft.fftshift(np.fft.fftfreq(line.shape[0], px_m))
I_fft = line_FFT

# Create frequency array
freq = freq_y_cpm
M = len(freq)
df = freq[1]-freq[0]  # Frequency resolution [1/m]

mask = (freq > 0.12e6) & (freq < 0.17e6)

# Find peaks in FFT spectrum (use magnitude)
I_fft_magnitude = np.abs(I_fft)
# peaks, properties = find_peaks(I_fft_magnitude*mask, height=np.max(I_fft_magnitude*mask)*0.1)

# Identify carrier frequency (should be at ±1/pitch)
carrier_freq_ideal = grating_freq_cpm
#carrier_idx = peaks[np.argmin(np.abs(freq[peaks] - carrier_freq_ideal))]
carrier_idx = np.argmax(I_fft_magnitude*mask)
carrier_freq_measured = freq[carrier_idx]


# Create Gaussian filter centered on +1 order
filter_width = 10.0  # pixels (standard deviation)
x_indices = np.arange(M)
gaussian_filter = np.exp(-0.5 * ((x_indices - carrier_idx) / filter_width)**2)

# Apply filter to FFT
I_fft_filtered = I_fft * gaussian_filter

# Normalize for visualization (so all curves are visible on same plot)
I_fft_magnitude_norm = I_fft_magnitude / np.max(I_fft_magnitude)
I_fft_filtered_norm = np.abs(I_fft_filtered) / np.max(np.abs(I_fft_magnitude))

# Inverse FFT to get complex field
complex_field = np.fft.ifft(np.fft.ifftshift(I_fft_filtered))

# Extract amplitude and wrapped phase
amplitude = np.abs(complex_field)
phase_wrapped = np.angle(complex_field)

# Standard unwrap (left-to-right) and center at x=0
phase_unwrapped = np.unwrap(phase_wrapped)
phase_centered = phase_unwrapped - phase_unwrapped[M//2]

dy_m = x_m[1] - x_m[0]  # Spatial sampling interval [m]

# Simple cumulative integration with scaling
wavefront_m = np.cumsum(phase_centered) * dy_m / grating_pitch_m * wavelength_m / 2

# Re-center at x=0 to remove piston term
wavefront_reconstructed = wavefront_m - wavefront_m[M//2]
wavefront_wave = wavefront_reconstructed # Convert to meters


plt.plot(x_m*1e3, wavefront_reconstructed*1e9)
plt.title('Reconstructed wavefront')
plt.xlabel('Position [mm]')
plt.ylabel('Wavefront (nm)')
plt.show()

In [ ]:
# Simple cumulative integration with scaling
wavefront_m = np.cumsum(phase_centered) * dy_m / grating_pitch_m * wavelength_m / 2

# Re-center at x=0 to remove piston term
wavefront_reconstructed = wavefront_m - wavefront_m[M//2]
wavefront_rad = wavefront_reconstructed/wavelength_m * 2 * np.pi # Convert to radians


plt.plot(x_m*1e3, wavefront_rad)
plt.title('Reconstructed wavefront')
plt.xlabel('Position [mm]')
plt.ylabel('Wavefront (rad)')
plt.show()

In [ ]:
def estimate_wavefront(line_profile):
    #other useful metadata:
    grating_pitch_m = 7e-6  # 
    chip_ccd_distance = 300e-3


    grating_freq_cpm = 1/grating_pitch_m  # cycles per micron for 7 um pitch
    wavelength_m = 1.24e-6 / (energy_eV[idx])  # Convert eV to wavelength in meters
    px_m = effective_pixel_size_um * 1e-6  # Convert effective pixel size to meters

    line = line_profile
    line_FFT = np.fft.fftshift(np.fft.fft(line, axis=0), axes=0)
    freq_y_cpm = np.fft.fftshift(np.fft.fftfreq(line.shape[0], px_m))
    I_fft = line_FFT

    # Create frequency array
    freq = freq_y_cpm
    M = len(freq)
    df = freq[1]-freq[0]  # Frequency resolution [1/m]

    mask = (freq > 0.12e6) & (freq < 0.17e6)

    # Find peaks in FFT spectrum (use magnitude)
    I_fft_magnitude = np.abs(I_fft)
    # peaks, properties = find_peaks(I_fft_magnitude*mask, height=np.max(I_fft_magnitude*mask)*0.1)

    # Identify carrier frequency (should be at ±1/pitch)
    carrier_freq_ideal = grating_freq_cpm
    #carrier_idx = peaks[np.argmin(np.abs(freq[peaks] - carrier_freq_ideal))]
    carrier_idx = np.argmax(I_fft_magnitude*mask)
    carrier_freq_measured = freq[carrier_idx]


    # Create Gaussian filter centered on +1 order
    filter_width = 10.0  # pixels (standard deviation)
    x_indices = np.arange(M)
    gaussian_filter = np.exp(-0.5 * ((x_indices - carrier_idx) / filter_width)**2)

    # Apply filter to FFT
    I_fft_filtered = I_fft * gaussian_filter

    # Normalize for visualization (so all curves are visible on same plot)
    I_fft_magnitude_norm = I_fft_magnitude / np.max(I_fft_magnitude)
    I_fft_filtered_norm = np.abs(I_fft_filtered) / np.max(np.abs(I_fft_magnitude))

    # Inverse FFT to get complex field
    complex_field = np.fft.ifft(np.fft.ifftshift(I_fft_filtered))

    # Extract amplitude and wrapped phase
    amplitude = np.abs(complex_field)
    phase_wrapped = np.angle(complex_field)

    # Standard unwrap (left-to-right) and center at x=0
    phase_unwrapped = np.unwrap(phase_wrapped)
    phase_centered = phase_unwrapped - phase_unwrapped[M//2]

    dy_m = x_m[1] - x_m[0]  # Spatial sampling interval [m]

    # Simple cumulative integration with scaling
    wavefront_m = np.cumsum(phase_centered) * dy_m / grating_pitch_m * wavelength_m / 2

    # Re-center at x=0 to remove piston term
    wavefront_reconstructed = wavefront_m - wavefront_m[M//2]
    wavefront_wave = wavefront_reconstructed # Convert to meters

    return wavefront_wave



In [ ]:
for idx in range(22,32):
    line_profile = images[idx][:,495]
    wavefront = estimate_wavefront(line_profile)
    plt.plot(x_m*1e3, wavefront*1e9, label='Energy = {:1.1f} eV, bender = {:1.1f} um'.format(energy_eV[idx], bender_um[idx]))
plt.title('Reconstructed wavefront')
plt.xlabel('Position [mm]')
plt.ylabel('Wavefront (nm)')
plt.legend()
plt.show()

In [ ]:
img_list = []
for index in range(22,32):
    img_list.append(images[index])

# Stack the images horizontally
stacked_image = np.hstack(img_list)
plt.imshow(stacked_image, cmap='inferno')
plt.show()

